# 02 — Reward Model Analysis

After `make train-rm` completes, this notebook:
- Plots training curves (loss and accuracy by epoch)
- Visualises the reward score distribution for chosen vs rejected responses
- Computes per-length accuracy to check if the RM is just a length heuristic
- Shows the highest- and lowest-confidence examples

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk
from torch.utils.data import DataLoader

from src.reward_model import RewardModel, reward_accuracy

In [ ]:
# Load training history
with open('../checkpoints/rm/training_history.json') as f:
    history = json.load(f)

epochs = [h['epoch'] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, [h['train_loss'] for h in history], 'o-', label='train')
ax1.plot(epochs, [h['val_loss']   for h in history], 's--', label='val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Bradley-Terry loss')
ax1.set_title('RM Training Loss')
ax1.legend()

ax2.plot(epochs, [h['train_acc'] for h in history], 'o-', label='train')
ax2.plot(epochs, [h['val_acc']   for h in history], 's--', label='val')
ax2.axhline(0.65, color='green', linestyle=':', label='65% target')
ax2.axhline(0.5,  color='red',   linestyle=':', label='random')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('RM Val Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('../results/figures/02_rm_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Load model and score the val split
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RewardModel.from_checkpoint('../checkpoints/rm/best_rm.pt').to(device).eval()

dataset = load_from_disk('../data/processed')
val = dataset['val']

chosen_scores, rejected_scores = [], []

with torch.no_grad():
    for i in range(0, min(len(val), 1000), 8):  # first 1000 examples, batch=8
        batch = val.select(range(i, min(i+8, len(val))))
        def to_tensor(key):
            seqs = [torch.tensor(x, dtype=torch.long) for x in batch[key]]
            max_len = max(s.shape[0] for s in seqs)
            ids = torch.zeros(len(seqs), max_len, dtype=torch.long)
            mask = torch.zeros(len(seqs), max_len, dtype=torch.long)
            for j, s in enumerate(seqs):
                ids[j, :s.shape[0]] = s
                mask[j, :s.shape[0]] = 1
            return ids.to(device), mask.to(device)
        
        cids, cmask = to_tensor('chosen_input_ids')
        rids, rmask = to_tensor('rejected_input_ids')
        chosen_scores.extend(model(cids, cmask).cpu().tolist())
        rejected_scores.extend(model(rids, rmask).cpu().tolist())

chosen_scores  = np.array(chosen_scores)
rejected_scores = np.array(rejected_scores)
acc = (chosen_scores > rejected_scores).mean()
print(f"Val accuracy on {len(chosen_scores)} examples: {acc:.4f}")

In [ ]:
# Score distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(chosen_scores,   bins=50, alpha=0.7, label='chosen',   color='steelblue')
ax1.hist(rejected_scores, bins=50, alpha=0.7, label='rejected', color='salmon')
ax1.set_xlabel('Reward score')
ax1.set_ylabel('Count')
ax1.set_title('Reward Score Distributions')
ax1.legend()

margin = chosen_scores - rejected_scores
ax2.hist(margin, bins=50, color='mediumpurple', alpha=0.8)
ax2.axvline(0, color='black', linestyle='--')
ax2.set_xlabel('r_w − r_l  (margin)')
ax2.set_title(f'Score margin  (acc = {acc:.3f})')

plt.tight_layout()
plt.savefig('../results/figures/02_rm_score_distributions.png', dpi=150)
plt.show()